# 04_join_demografia_ufpa

Consolidacao das bases demografia, UFPA produtiva/territorio e UFPA renda/atividade.


## 1. Objetivo e granularidade

A base `df_caf_demografia` está em **nível pessoa** e a base `df_caf_ufpa_produtiva_territorio` em **nível família**.

Ao fazer `left join` por `id_familia`, os atributos de família são repetidos para todas as pessoas da mesma família. Este comportamento é esperado e deve ser explicitado para evitar interpretações incorretas.

## 2. Imports e configuração

Definimos caminhos, parâmetros de exibição e validamos a existência dos arquivos de entrada.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DEMOGRAFIA_PATH = ROOT / 'dados' / 'dados_processados' / 'df_caf_demografia.parquet'
UFPA_PATH = ROOT / 'dados' / 'dados_processados' / 'df_caf_ufpa_produtiva_territorio.parquet'
RENDA_ATIVIDADE_PATH = ROOT / 'dados' / 'dados_processados' / 'df_caf_ufpa_renda_atividade.parquet'
OUTPUT_PATH = ROOT / 'dados' / 'dados_processados' / 'df_caf_demografia_ufpa_consolidado.parquet'

for path in [DEMOGRAFIA_PATH, UFPA_PATH, RENDA_ATIVIDADE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {path}')

print(f'ROOT: {ROOT}')
print(f'DEMOGRAFIA_PATH: {DEMOGRAFIA_PATH}')
print(f'UFPA_PATH: {UFPA_PATH}')
print(f'RENDA_ATIVIDADE_PATH: {RENDA_ATIVIDADE_PATH}')
print(f'OUTPUT_PATH: {OUTPUT_PATH}')


## 3. Leitura dos dados

Carregamos os datasets já processados e inspecionamos rapidamente a estrutura inicial.

In [ ]:
df_caf_demografia = pd.read_parquet(DEMOGRAFIA_PATH)
df_caf_ufpa = pd.read_parquet(UFPA_PATH)
df_caf_ufpa_renda_atividade = pd.read_parquet(RENDA_ATIVIDADE_PATH)

print(f'Shape demografia: {df_caf_demografia.shape}')
print(f'Shape UFPA produtiva/territorio: {df_caf_ufpa.shape}')
print(f'Shape UFPA renda/atividade: {df_caf_ufpa_renda_atividade.shape}')

print('\nAmostra demografia:')
display(df_caf_demografia.head(3))

print('Amostra UFPA produtiva/territorio:')
display(df_caf_ufpa.head(3))

print('Amostra UFPA renda/atividade:')
display(df_caf_ufpa_renda_atividade.head(3))


## 4. Diagnóstico inicial das chaves

Validamos `id_familia` e verificamos duplicidades em chaves relevantes antes do join.

In [ ]:
for df_ref in [df_caf_demografia, df_caf_ufpa, df_caf_ufpa_renda_atividade]:
    if 'id_familia' in df_ref.columns:
        df_ref['id_familia'] = df_ref['id_familia'].astype('string').str.strip()

if 'ibge' in df_caf_demografia.columns:
    df_caf_demografia['ibge'] = df_caf_demografia['ibge'].astype('string').str.zfill(7)

print('Missing de id_familia:')
print(f"- Demografia:      {int(df_caf_demografia['id_familia'].isna().sum()):,}")
print(f"- UFPA territorio: {int(df_caf_ufpa['id_familia'].isna().sum()):,}")
print(f"- UFPA renda:      {int(df_caf_ufpa_renda_atividade['id_familia'].isna().sum()):,}")

if 'cpf_agricultor' in df_caf_demografia.columns:
    duplicados_cpf = int(df_caf_demografia.duplicated(subset=['cpf_agricultor']).sum())
    print(f'Duplicatas de cpf_agricultor na demografia: {duplicados_cpf:,}')

duplicados_familia_ufpa = int(df_caf_ufpa.duplicated(subset=['id_familia']).sum())
print(f'Duplicatas de id_familia na UFPA territorio: {duplicados_familia_ufpa:,}')
if duplicados_familia_ufpa > 0:
    print('Removendo duplicatas de id_familia na UFPA territorio (keep=first).')
    df_caf_ufpa = df_caf_ufpa.drop_duplicates(subset=['id_familia'], keep='first').copy()

duplicados_familia_renda = int(df_caf_ufpa_renda_atividade.duplicated(subset=['id_familia']).sum())
print(f'Duplicatas de id_familia na UFPA renda/atividade: {duplicados_familia_renda:,}')
if duplicados_familia_renda > 0:
    print('Removendo duplicatas de id_familia na UFPA renda/atividade (keep=first).')
    df_caf_ufpa_renda_atividade = df_caf_ufpa_renda_atividade.drop_duplicates(subset=['id_familia'], keep='first').copy()


## 5. Cobertura de chaves antes do join

Medimos quantas famílias da demografia encontram correspondência na base UFPA.

In [ ]:
familias_demografia = df_caf_demografia[['id_familia']].drop_duplicates()
familias_ufpa = df_caf_ufpa[['id_familia']].drop_duplicates().assign(st_tem_ufpa=1)
familias_renda = df_caf_ufpa_renda_atividade[['id_familia']].drop_duplicates().assign(st_tem_renda=1)

cobertura_familias = familias_demografia.merge(
    familias_ufpa,
    on='id_familia',
    how='left',
)
cobertura_familias = cobertura_familias.merge(
    familias_renda,
    on='id_familia',
    how='left',
)

cobertura_familias['st_tem_ufpa'] = cobertura_familias['st_tem_ufpa'].fillna(0).astype('int8')
cobertura_familias['st_tem_renda'] = cobertura_familias['st_tem_renda'].fillna(0).astype('int8')

total_familias = len(cobertura_familias)
familias_com_ufpa = int(cobertura_familias['st_tem_ufpa'].sum())
familias_com_renda = int(cobertura_familias['st_tem_renda'].sum())
familias_com_ambos = int(((cobertura_familias['st_tem_ufpa'] == 1) & (cobertura_familias['st_tem_renda'] == 1)).sum())

print(f'Total de familias na demografia: {total_familias:,}')
print(f'Familias com correspondencia na UFPA territorio: {familias_com_ufpa:,} ({familias_com_ufpa/total_familias:.2%})')
print(f'Familias com correspondencia na UFPA renda/atividade: {familias_com_renda:,} ({familias_com_renda/total_familias:.2%})')
print(f'Familias com correspondencia nas duas bases: {familias_com_ambos:,} ({familias_com_ambos/total_familias:.2%})')


## 6. Left join

Executamos o `left join` por `id_familia`, preservando todas as pessoas da base demográfica.

In [ ]:
linhas_demografia = len(df_caf_demografia)

colunas_renda = [
    c for c in df_caf_ufpa_renda_atividade.columns
    if c not in {'id_familia', 'ibge'}
]

df = (
    df_caf_demografia
    .merge(
        df_caf_ufpa_renda_atividade[['id_familia'] + colunas_renda],
        on='id_familia',
        how='left',
        validate='m:1',
    )
    .merge(
        df_caf_ufpa,
        on='id_familia',
        how='left',
        validate='m:1',
    )
)

# Colunas da base de renda/atividade posicionadas a esquerda no dataframe final
colunas_esquerda = ['id_familia'] + colunas_renda
df = df[colunas_esquerda + [c for c in df.columns if c not in colunas_esquerda]]

print(f'Linhas na demografia (antes do join): {linhas_demografia:,}')
print(f'Linhas no consolidado (apos joins): {len(df):,}')
print(f'Colunas no consolidado: {len(df.columns)}')


## 7. Padronização de tipos pós-join

Após o join, colunas inteiras da UFPA podem virar `float` por causa de `NaN`.
Aqui normalizamos os tipos para manter consistência e eficiência de memória.

In [ ]:
for col in ['ibge', 'cpf_agricultor', 'id_familia']:
    if col in df.columns:
        df[col] = df[col].astype('string')

int8_nullable_cols = [
    # UFPA territorio
    'cd_uf', 'id_regiao', 'nr_modulo_fiscal',
    'n_imoveis_total', 'n_imoveis_ha', 'n_imoveis_m2', 'n_imoveis_m3',
    'tem_incra', 'st_geo_valida',
    # UFPA renda/atividade
    'n_tipos_renda', 'n_produtos',
    'tem_agroecologia', 'tem_agroindustria', 'tem_pecuaria', 'st_possui_renda',
]

for col in int8_nullable_cols:
    if col in df.columns:
        serie = pd.to_numeric(df[col], errors='coerce')
        max_abs = serie.abs().max(skipna=True)
        if pd.notna(max_abs) and max_abs > 127:
            raise ValueError(f'{col} excede limite do Int8: max_abs={max_abs}')
        df[col] = serie.astype('Int8')

float32_cols = [
    # UFPA territorio
    'lat_imovel_principal', 'lon_imovel_principal',
    'soma_area_ha', 'soma_area_m2', 'soma_volume_m3',
    # UFPA renda/atividade
    'renda_total_estimada', 'renda_total_auferida',
    'renda_dentro_estimada', 'renda_dentro_auferida',
    'renda_fora_estimada', 'renda_fora_auferida',
    'renda_top1_produto_estimada', 'share_top1_produto_estimada',
]

for col in float32_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

print('Dtypes principais apos padronizacao:')
display(df[['id_familia'] + [c for c in int8_nullable_cols if c in df.columns]].dtypes)


## 8. Validação pós-join

Checamos cobertura dos atributos UFPA e coerência básica das variáveis de enquadramento de renda.

In [ ]:
if 'cd_uf' in df.columns:
    df['st_tem_ufpa'] = df['cd_uf'].notna().astype('int8')
else:
    df['st_tem_ufpa'] = 0

if 'renda_total_estimada' in df.columns:
    df['st_tem_renda_atividade'] = df['renda_total_estimada'].notna().astype('int8')
else:
    df['st_tem_renda_atividade'] = 0

print('Distribuicao de st_tem_ufpa:')
display(df['st_tem_ufpa'].value_counts(dropna=False).sort_index())

print('Distribuicao de st_tem_renda_atividade:')
display(df['st_tem_renda_atividade'].value_counts(dropna=False).sort_index())

if 'soma_area_ha' in df.columns:
    print('Quantis de soma_area_ha:')
    display(df['soma_area_ha'].quantile([0.50, 0.90, 0.99, 0.999, 0.9999]))

if 'renda_total_estimada' in df.columns:
    print('Quantis de renda_total_estimada:')
    display(df['renda_total_estimada'].quantile([0.50, 0.90, 0.99, 0.999, 0.9999]))

colunas_enquadramento = ['A', 'AC', 'B', 'V', 'Desconhecido']
if set(colunas_enquadramento).issubset(df.columns):
    soma_enquadramento = df[colunas_enquadramento].sum(axis=1)
    print('Distribuicao da soma das dummies de enquadramento:')
    display(soma_enquadramento.value_counts(dropna=False).sort_index())
    inconsist_enquadramento = int((soma_enquadramento != 1).sum())
    print(f'Registros com soma das dummies diferente de 1: {inconsist_enquadramento:,}')

print('Shape final consolidado:')
print(df.shape)
df.info(show_counts=True)


In [ ]:
df

## 9. Persistência

Salvamos o dataset consolidado em parquet para as próximas etapas de modelagem.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)

print(f'Arquivo salvo em: {OUTPUT_PATH}')
print(f'Shape final salvo: {df.shape}')
print(f'Total de colunas: {len(df.columns)}')